# VAT (ICMS) collection forecast: the closed v1 protocol on the total, Declared (Declarado), and Assessed (Autuado)

**Question.** On each v1 grain, which model has the lowest test sMAPE for total VAT (ICMS), for Declared, and for Assessed?

**Protocol.** This notebook keeps the closed v1 benchmark. One contiguous train, validation, and test cut. No rolling origin. No MinT reconciliation. No new model family. Models are M0–M9 and E1–E4. The rank is test sMAPE. Hyperparameters are chosen again on the validation block of the series in hand, with the same grids as v1.

The model functions are the v1 notebook cells, copied in `projects/monitoramento/scripts/v1_protocol_extract.py`. The fit is `run_v1_segment_benchmark.py`. This notebook reads the saved forecasts. It does not refit them.

**Sample.** Payments and stock end on 31 December 2025. A week label after that date is dropped. No test window contains a later date.

**Universe.** Legal-entity ICMS in the lab extract. Assessed (Autuado) is the type `ICMS Autuação`. Declared (Declarado) is `ICMS Declarado`. Total is every ICMS type in that extract.

## sMAPE and the skill score

Let \(y_t\) be collection and \(\hat y_t\) the forecast. Let \(\epsilon\) be a small positive tolerance. sMAPE averages the symmetric percentage error on times with \(|y_t| + |\hat y_t| > \epsilon\) (Hyndman and Athanasopoulos, 2021, *Forecasting: Principles and Practice*, section 5.8):

\[
\mathrm{sMAPE}
= \frac{100}{n_S}
\sum_{t:\,|y_t|+|\hat y_t|>\epsilon}
\frac{2\,|y_t-\hat y_t|}{|y_t|+|\hat y_t|}.
\]

The official rank on a grain is test sMAPE. A lower value is better.

The skill score is the Murphy (1988) ratio. \(A\) is the test sMAPE of a model. \(A_{\mathrm{ref}}\) is the test sMAPE of the seasonal naive on the same grain:

\[
\mathrm{SS}_{\mathrm{sMAPE}} = 1 - \frac{A}{A_{\mathrm{ref}}}.
\]

A positive skill score means lower test sMAPE than the seasonal naive. The seasonal lag is 12 (monthly), 7 (daily A), 5 (daily B), and 52 (weekly). The skill score does not replace the rank.

## Diebold-Mariano test with the HLN correction

Diebold and Mariano (1995, doi:10.1080/07350015.1995.10524599) test equal predictive accuracy. The loss here is the sMAPE summand, so the test uses the same score as the rank. Let \(d_t = L(y_t, \hat y^{\mathrm{model}}_t) - L(y_t, \hat y^{\mathrm{naive}}_t)\). A negative mean favors the model.

For horizon \(h=1\), the Newey-West truncation has no lag. Harvey, Leybourne and Newbold (1997, doi:10.1016/S0169-2070(96)00719-4) multiply the DM statistic by \(\sqrt{(n-1)/n}\) and compare it with a Student-\(t\) distribution on \(n-1\) degrees of freedom. The test is two-sided.

This horizon is the one-step correction. It stays defined when the test length equals the multi-step horizon. On that case the HLN factor for \(h=n\) is not positive, so \(h=n\) is not used.

## Model confidence set

Hansen, Lunde and Nason (2011, doi:10.3982/ECTA5771) build a set of models that the data do not separate at a chosen level. This notebook uses level 0.10. The statistic is the maximum \(t\) of each model against the average loss of the others. A circular block bootstrap with 999 draws supplies the p-value. The block length is the cube root of the test length. A model leaves the set when the equivalence test rejects. The reported p-value is the running maximum of the elimination p-values, as in the paper.

The set test and the pairwise DM test answer different questions. The DM test compares the best model with the seasonal naive. The set test asks which models remain after sequential elimination. On a short test block the set is wide.

## Deflation

v1 does not deflate the collection target. The daily file in v1 is `SUM(VALOR_TOTAL_GARE)`. The v1 notebook and the v1 data section contain no price index.

`VALOR_TOTAL_GARE` is the cash amount on the slip. In this extract, principal plus interest plus fine is about 83 percent of that total. The extract does not store the remaining component. The ratio is similar for every ICMS type. That gap is a missing component, not a deflator.

This benchmark uses the same field. It does not divide by an IPCA index. A second price adjustment would not match v1, and it would adjust a series that is already a nominal cash sum.

The monthly stock covariate is `VALOR_SEM_HONORARIOS` from the snapshot of that month. It is not restated to a base-year price. The IPCA deflator in the lab applies to two macro series (GDP and credit to firms). It does not apply to GARE or to the enrolled balance in this protocol.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "projects").exists():
    ROOT = next(p for p in [ROOT, *ROOT.parents] if (p / "projects").exists())
ART = ROOT / "projects/monitoramento/notebooks/articles/arrecadacao_forecast_v1_segmentado"
sys.path.insert(0, str(ROOT / "projects/monitoramento/scripts"))

stock = pd.read_csv(ART / "data/share_stock_2025_12.csv")
coll = pd.read_csv(ART / "data/share_collection_through_2025.csv")
stock["share_pct"] = 100 * stock["share"]
coll["share_pct"] = 100 * coll["share_valor_total_gare"]
print("December 2025 stock share")
print(stock[["tipo", "stock", "n_rows", "share_pct"]].to_string(index=False))
print("Collection share through 2025-12-31")
print(coll[["tipo", "valor_sum", "share_pct", "parts_over_total"]].to_string(index=False))

**How to read the share tables.** `share_pct` is the percent of legal-entity ICMS value. Stock is the enrolled balance in the December 2025 snapshot. Collection is the sum of `VALOR_TOTAL_GARE` from 4 January 2016 through 31 December 2025. `parts_over_total` is (principal + interest + fine) / slip total. A value near 0.83 means the slip total is larger than those three parts. It is not an index deflator.

Declared (Declarado) and Assessed (Autuado) are the two types in the forecast. The other three types are a fraction of one percent of stock and of collection. That is the reason the benchmark stops at these two segments. Assessed is most of the stock. Declared is most of the collection. The two ranks need not match.

In [ ]:
comp_path = ART / "results/comparison.csv"
if comp_path.exists():
    comp = pd.read_csv(comp_path)
    cols = ["series", "grain", "test_start", "test_end", "best", "smape", "seasonal_smape", "skill", "dm_hln", "dm_p", "best_in_mcs"]
    print(comp[cols].to_string(index=False))
else:
    print("comparison.csv is not written yet")
detail_path = ART / "results/comparison.json"
if detail_path.exists():
    detail = json.loads(detail_path.read_text())
    for row in detail:
        included = ", ".join(row["mcs_included"])
        print(f"{row['series']} {row['grain']}: MCS = {included}")

**How to read the comparison.** Each row is one series and one grain. `best` is the lowest test sMAPE in the ranked file. `skill` uses the seasonal naive of that grain. `dm_hln` and `dm_p` are the HLN test of that best model against the seasonal naive. `best_in_mcs` is true when the best model remains in the 90 percent model confidence set. The lines under the table name every model that remains in that set.

Do not read a change of the best model across series as a unique winner when the confidence set still contains other models. On a test block of 12 or 13 points the set is wide.

**How to read the figures.**
The notebook writes every manuscript figure from the saved forecasts.
It does not refit a model.
Teal marks the lowest test sMAPE.
A forecast line keeps one color for that model in every figure.
Total is teal, Declared is purple, and Assessed is orange when the series itself is the category.
The time plot is the test block.
A second dashed line appears only when its test sMAPE is within 5 percent of the leader.
The scatter is the same block, with a 45-degree line.
A point below that line is an under-forecast.


In [ ]:
from IPython.display import Image, Markdown, display

from make_segment_figures import main as make_figures
from make_segment_figures import manuscript_dir

written = make_figures()
order = [
    "fig_shares",
    "fig_smape_total",
    "fig_smape_declarado",
    "fig_smape_autuado",
    "fig_skill_total",
    "fig_skill_declarado",
    "fig_skill_autuado",
    "fig_ablation_smape",
    "fig_ablation_skill",
    "fig_pred_monthly",
    "fig_scatter_monthly",
    "fig_pred_weekly",
    "fig_scatter_weekly",
    "fig_pred_business_daily",
    "fig_scatter_business_daily",
    "fig_pred_daily",
    "fig_scatter_daily",
    "fig_residual_main",
    "fig_residual_daily",
]
fig_dir = manuscript_dir() / "figures"
shown = set()
for stem in order:
    png = fig_dir / f"{stem}.png"
    display(Markdown(f"**{stem}**"))
    display(Image(filename=str(png)))
    shown.add(stem)
for pdf in written:
    if pdf.stem in shown:
        continue
    display(Markdown(f"**{pdf.stem}**"))
    display(Image(filename=str(pdf.with_suffix('.png'))))
